# Python Virtual Environments
### Using `venv` and `poetry`

A **virtual environment** is an isolated Python installation for a specific project.  
It keeps your project's dependencies separate from your system Python and from other projects.


This notebook covers two tools:

| Tool | Built-in? | Best for |
|------|-----------|----------|
| `venv` | Yes (stdlib) | Simple projects, no extra installs needed |
| `poetry` | Needs install | Projects with complex deps, packaging, publishing |

## `venv` (built into Python)

`venv` ships with Python 3.3+ and is the standard, zero-dependency approach.

### Create a virtual environment

Run this in your **terminal** (not inside Python):

```bash
# macOS / Linux
python3 -m venv .venv

# Windows
py -m venv .venv
```

On Windows, `py` is the Python launcher. If `py` is unavailable, use `python -m venv .venv`.

This creates a `.venv/` folder in your current directory containing:
- A copy (or symlink) of the Python interpreter
- `pip` and `setuptools`
- An isolated `site-packages/` directory

### Activate the environment

Activation makes your terminal use *this* Python instead of the system one.

```bash
# macOS / Linux
source .venv/bin/activate

# Windows (Command Prompt)
.venv\Scripts\activate.bat
```

After activation your prompt changes to show the environment name:
```
(.venv) your-machine:your-project $
```

You can confirm which Python is active:

In [ ]:
# Run this cell to see which Python interpreter is currently in use
import sys
print(sys.executable)
print(f"Python version: {sys.version}")

### Install packages

With the environment active, `pip` installs **only into this environment**:

```bash
pip install numpy pandas matplotlib

# Install a specific version
pip install pandas==2.1.0

# Install from a requirements file
pip install -r requirements.txt
```

### Freeze and share your environment

```bash
# Save all installed packages + exact versions to a file
pip freeze > requirements.txt
```

A `requirements.txt` looks like:
```
numpy==1.26.4
pandas==2.1.0
matplotlib==3.8.2
```

Anyone can recreate your environment with:
```bash
python -m venv .venv

# macOS / Linux
source .venv/bin/activate

# Windows (Command Prompt)
.venv\Scripts\activate.bat
```

### Deactivate and delete

```bash
# Leave the virtual environment (go back to system Python)
deactivate

# Delete the environment entirely (it's just a folder)
rm -rf .venv                      # macOS / Linux
rmdir /s /q .venv                 # Windows (Command Prompt)
```

> **Always add `.venv/` to your `.gitignore`** — never commit the environment folder itself, only `requirements.txt`.

### `venv` — Quick Reference

```
python -m venv .venv                # create
source .venv/bin/activate           # activate (macOS/Linux)
.venv\Scripts\activate.bat        # activate (Windows cmd)

pip install <package>               # install
pip freeze > requirements.txt       # save deps
deactivate                          # exit
```

## `poetry`

`poetry` is a modern dependency manager and packaging tool.  
It goes beyond `venv` + `pip` by also handling:
- **Dependency resolution** (avoids conflicting sub-dependencies)
- **Lock files** (`poetry.lock`) for exact reproducibility
- **Building and publishing** packages to PyPI
- **Project metadata** in a single `pyproject.toml`

### Install Poetry

Install it **once** on your system (outside any virtual environment):

```bash
# Official installer (Linux, macOS, Windows WSL)
curl -sSL https://install.python-poetry.org | python3 -
# curl -sSL https://install.python-poetry.org | /usr/local/bin/python3 -

# macOS with Homebrew
brew install poetry
```

If `poetry` is not recognized on Windows after installation, restart your terminal and ensure `%APPDATA%\Python\Scripts` is on your `PATH`.

Verify the installation:

In [ ]:
# Check poetry is installed and see its version
import subprocess
result = subprocess.run(["poetry", "--version"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "Poetry not found — install it first (see cell above)")

### Create a new project

```bash
# Creates a new folder with project scaffolding
poetry new my-project
```

This generates:
```
my-project/
├── pyproject.toml      ← project metadata + dependencies
├── README.md
├── my_project/
│   └── __init__.py
└── tests/
    └── __init__.py
```

**Adding Poetry to an existing project:**
```bash
cd my-existing-project
poetry init        # interactive setup, creates pyproject.toml
```

### The `pyproject.toml` file

This is the single source of truth for your project. A typical file looks like:

```toml
[tool.poetry]
name = "my-project"
version = "0.1.0"
description = "A short description"
authors = ["Your Name <you@example.com>"]

[tool.poetry.dependencies]
python = "^3.11"
pandas = "^2.1"
numpy = "^1.26"

[tool.poetry.group.dev.dependencies]
pytest = "^7.4"
black = "^23.0"

[build-system]
requires = ["poetry-core"]
build-backend = "poetry.core.masonry.api"
```

> **Version constraints:**  
> `^2.1` means `>=2.1.0, <3.0.0` (compatible releases)  
> `~2.1` means `>=2.1.0, <2.2.0` (more restrictive)  
> `*` means any version

### Add and remove dependencies

```bash
# Add a package (updates pyproject.toml + poetry.lock automatically)
poetry add pandas
poetry add pandas@^2.1          # specific constraint

# Add a dev-only dependency (not needed in production)
poetry add --group dev pytest
poetry add --group dev black mypy

# Remove a package
poetry remove pandas
```

### Install and sync the environment

```bash
# Install all dependencies from pyproject.toml (creates .venv automatically)
poetry install

# Install without dev dependencies (e.g. in production)
poetry install --without dev

# Sync exactly to poetry.lock (removes packages not in lockfile)
poetry install --sync
```

Poetry creates and manages the virtual environment for you — you don't need to run `python -m venv` manually.

### Run commands inside the environment

Two ways to use the environment:

```bash
# Option A: spawn a shell with the environment active
poetry shell
python my_script.py     # now running inside the venv
exit                    # leave the poetry shell

# Option B: run a single command without spawning a shell
poetry run python my_script.py
poetry run pytest
poetry run jupyter notebook
```

> `poetry run` is preferred in scripts and CI/CD pipelines since it doesn't require shell activation.

### The lock file

`poetry.lock` is auto-generated and records the **exact version of every package** (including all transitive dependencies).

```bash
# Regenerate the lock file (e.g. after manually editing pyproject.toml)
poetry lock

# Update all packages to latest allowed versions
poetry update

# Update a specific package only
poetry update pandas
```

| File | Commit to git? | Purpose |
|------|---------------|--------|
| `pyproject.toml` | Yes | Human-readable constraints |
| `poetry.lock` | Yes | Exact reproducible snapshot |
| `.venv/` | No | Generated files, add to `.gitignore` |

### Useful inspection commands

```bash
# Show all installed packages
poetry show

# Show the dependency tree
poetry show --tree

# Show where the virtual environment lives on disk
poetry env info

# List all environments for this project
poetry env list

# Remove the environment (to start fresh)
poetry env remove python
```

### `poetry` — Quick Reference

```
poetry new my-project          # scaffold new project
poetry init                    # add poetry to existing project
poetry add <package>           # add dependency
poetry add --group dev <pkg>   # add dev dependency
poetry install                 # install all deps from lockfile
poetry run python script.py    # run inside the environment
poetry shell                   # activate an interactive shell
poetry update                  # update deps to latest allowed
poetry show --tree             # inspect dependency tree
```

## `venv` vs `poetry` — When to use which

| Situation | Recommended tool |
|-----------|------------------|
| Quick script or experiment | `venv` + `pip` |
| Team project needing reproducibility | `poetry` |
| Building a library to publish to PyPI | `poetry` |
| CI/CD pipeline | `poetry` (lock file guarantees exact builds) |
| Learning Python for the first time | `venv` (fewer concepts at once) |
| Project with complex, conflicting deps | `poetry` (better resolver) |

They are **not mutually exclusive** — Poetry uses a `.venv` internally. The difference is that Poetry adds dependency resolution, lock files, and project management on top.

## Summary

```
venv                              poetry
─────────────────────────────     ──────────────────────────────────
python -m venv .venv              poetry new project / poetry init
source .venv/bin/activate         (automatic)
pip install pandas                poetry add pandas
pip freeze > requirements.txt     pyproject.toml + poetry.lock
pip install -r requirements.txt   poetry install
deactivate                        exit (from poetry shell)
```

On Windows, replace `source .venv/bin/activate` with `.venv\Scripts\activate.bat`.

Both tools solve the same core problem — isolation — but Poetry gives you much more control over dependency management and project lifecycle. For anything beyond a single-file script, Poetry is the modern standard.